In [1]:
# Import required libs
import pandas as pd
import numpy as np
import json
import os
from dotenv import load_dotenv
from minsearch import Index
from mistralai import Mistral

In [2]:
with open('documents.json', 'rt', encoding='utf-8') as f_out:
    raw_docs = json.load(f_out)

In [3]:
documents = []
for course_dict in raw_docs:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [4]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [5]:
# Embedding with minsearch
index = Index(
    text_fields = ['question', 'section', 'text' ],
    keyword_fields = ['course']
)

In [6]:
index.fit(documents)

In [7]:
q = 'the course has already started, can I still enroll?'

In [119]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant.\n
ANSWER the QUESTION based on the CONTEXT from the FAQ database. \n
Use only the facts from the CONTEXT when answering the QUESTION.\n
If the user's question doesn't contain in the FAQ database, please just kindly decline the requrest and reponse in a kind manner.\n
Don't add any other extra words.\n
QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    prompt = prompt_template.format(question=query, context=context)
    return prompt

In [176]:
def search(query):
    boost = {'question': 3.0, 'section' :0.5}
    results = index.search(
        query = query,
        filter_dict = {'course': 'data-engineering-zoomcamp'},
        boost_dict = boost,
        num_results = 5
    )
    return results

In [92]:
query = "How do I run Kafka?"
search_results = search(query)
prompt = build_prompt(query, search_results)

In [93]:
# Building Model
load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY_2')
client = Mistral(api_key=api_key)

In [94]:
# Getting the response from LLM
def llm(query):
    model = "mistral-small-2506"
    try:
        response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'user',
                'content': query
            },
        ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"An error occured: {e}")

In [95]:
results = llm(prompt)

In [96]:
print(results)

Based on the provided CONTEXT, here are the relevant instructions for running Kafka:

1. **For Java Kafka (producer/consumer/KStreams)**:
   In the project directory, run:
   ```
   java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java
   ```

2. **For Python Kafka**:
   - Create and activate a virtual environment:
     ```
     python -m venv env
     source env/bin/activate  # (MacOS/Linux) or env\Scripts\activate (Windows)
     ```
   - Install dependencies:
     ```
     pip install -r ../requirements.txt
     ```
   - If you encounter module errors, use:
     ```
     pip install kafka-python-ng
     ```
   - Ensure Docker images are running before executing Python files.

3. **For permission issues with scripts**:
   If you get a "Permission denied" error (e.g., for `build.sh`), run:
   ```
   chmod +x build.sh
   ```

No other CONTEXT sections provide additional instructions for running Kafka.


In [120]:
# final_llm

def chat_bot():
    print("--- The chatbot has started ---\n--- Please use quit or exit to terminate the Program ---\n")
    while True:
        query = input("You: ")
        if query != 'exit' and query != 'quit':
            search_results = search(query)
            prompt = build_prompt(query, search_results)
            answer = llm(prompt)
            print(f"Bot: {answer}\n ---")
        else:
            print("Bot: Good Bye!...")
            print("\n--- End of the Program ---")
            break

In [121]:
chat_bot()

--- The chatbot has started ---
--- Please use quit or exit to terminate the Program ---



You:  Hi


Bot: Hello! How can I assist you today?
 ---


You:  What is today?


Bot: I kindly decline your request as the information is not available in the FAQ database.
 ---


...

Bot: You need to navigate to the working directory that contains terraform configuration files, and then run the command.
 ---


You:  How can I build agent?


Bot: I kindly decline your request.
 ---


You:  exit


Bot: Good Bye!...

--- End of the Program ---


In [124]:
# Elastic Search

In [125]:
# Indexing the documents with elastic search?
from elasticsearch import Elasticsearch

In [128]:
# Building es client
es_client = Elasticsearch('http://localhost:9200')

In [129]:
es_client.info()

ObjectApiResponse({'name': '8fe22e71b9b2', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'mQvmHFweT5qkeiUeM0o56w', 'version': {'number': '8.17.6', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'dbcbbbd0bc4924cfeb28929dc05d82d662c527b7', 'build_date': '2025-04-30T14:07:12.231372970Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [131]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = 'course-questions-2'
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions-2'})

In [136]:
from tqdm.auto import tqdm

In [138]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 948/948 [00:03<00:00, 309.13it/s]


In [154]:
query = 'I just discovered the course. Can I still join?'

In [159]:
# Query

def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }
    response = es_client.search(index=index_name, body=search_query)
    result_docs = []
    for doc in response['hits']['hits']:
        result_docs.append(doc['_source'])

    return result_docs

In [160]:
results = elastic_search(query)

In [165]:
def rag(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt)

In [166]:
answer = rag(query)

In [167]:
print(answer)

Yes, even if you don't register, you're still eligible to submit the homeworks. Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.


In [168]:
# final_llm for elastic_search

def rag_bot():
    print("--- The chatbot has started ---\n--- Please use quit or exit to terminate the Program ---\n")
    while True:
        query = input("You: ")
        if query != 'exit' and query != 'quit':
            search_results = elastic_search(query)
            prompt = build_prompt(query, search_results)
            answer = llm(prompt)
            print(f"Bot: {answer}\n ---")
        else:
            print("Bot: Good Bye!...")
            print("\n--- End of the Program ---")
            break

In [169]:
rag_bot()

--- The chatbot has started ---
--- Please use quit or exit to terminate the Program ---



You:  I just discovered the course. Can I still join?


Bot: Yes, even if you don't register, you're still eligible to submit the homeworks. Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.
 ---


You:  The course started, Can I still join?


Bot: Yes, even if you don't register, you're still eligible to submit the homeworks. Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.
 ---


You:  Do you know Myanmar?


Bot: I'm sorry, but I don't have information about Myanmar in the provided context.
 ---


You:  exit


Bot: Good Bye!...

--- End of the Program ---
